# Chaos testing

The setup includes Toxiproxy for chaos testing.

Notebooks / agents can choose to connect to an endpoint without chaos and one with chaos:

- Clean path: `os.getenv("MODEL_BASE_URL_CLEAN")`
- Chaos path: `os.getenv("MODEL_BASE_URL_CHAOS")`

Toxiproxy admin API (add/remove toxics on the proxies below):

- `os.getenv("TOXIPROXY_URL")`

On the chaos path two different proxies can be configured:

- `edge_chaos`
- `provider_chaos_ollama`

The `edge_chaos` proxy operates between the notebook and LiteLLM:

 - Notebook -> MODEL_BASE_URL_CHAOS -> Toxiproxy -> `edge_chaos` -> LiteLLM (chaos) -> local or cloud hosted LLM

 Use this to emulate edge chaos and to see how the notebook and the Python client code handle network issues: endpoint not available, slow connections, ...

 The `provider_chaos_ollama` proxy acts between LiteLLM and Ollama:

 - Notebook -> MODEL_BASE_URL_CHAOS -> Toxiproxy (-> edge_chaos without toxics) -> LiteLLM (chaos) -> `provider_chaos_ollama` -> ollama

 This proxy helps to check LiteLLM handling of network issues with regard to retry and fallback mechanisms. Note that the proxy applies to locally deployed LLMs only. However, since this is about testing the connection and not the inference, it makes no difference which LLM you put behind. 

 The notebook demonstrates the `edge_chaos` path. 

In [2]:
import os
from contextlib import contextmanager
from time import perf_counter

import httpx
import openai
import requests

# --- Config ---
API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError(
        "Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation."
    )

# clean and chaos ingress
BASE_URL_CLEAN = os.getenv("MODEL_BASE_URL_CLEAN")
BASE_URL_CHAOS = os.getenv("MODEL_BASE_URL_CHAOS")
if not BASE_URL_CLEAN:
    raise RuntimeError(
        "Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates."
    )
if not BASE_URL_CHAOS:
    raise RuntimeError(
        "Missing MODEL_BASE_URL_CHAOS in environment. Restart the dev container after env updates."
    )

# local CA from your compose cert setup
CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
VERIFY_CONFIG = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True

# toxiproxy admin API
TOXI_URL = os.getenv("TOXIPROXY_URL")
if not TOXI_URL:
    raise RuntimeError(
        "Missing TOXIPROXY_URL in environment. Restart the dev container after env updates."
    )
TOXI_URL = TOXI_URL.rstrip("/")

# The name of the proxy to use. Supported proxies:
# - edge_chaos
# - provider_chaos_ollama
# Edge only in this demo.
PROXY_NAME = "edge_chaos"
TOXIC_NAME = "nb_edge_latency"

# llm request config
MODEL = "ollama_chat/llama3.2:3b"
PROMPT = "Sag kurz Hallo auf Deutsch."


def make_client(base_url: str) -> openai.OpenAI:
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    return openai.OpenAI(
        api_key=API_KEY,
        base_url=base_url,
        http_client=http_client,
    )


def timed_chat(base_url: str, model: str = MODEL, prompt: str = PROMPT):
    client = make_client(base_url)
    start = perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = perf_counter() - start
    text = response.choices[0].message.content or ""
    return elapsed, text


def remove_toxic_if_present():
    # Ignore errors if toxic does not exist yet
    try:
        requests.delete(
            f"{TOXI_URL}/proxies/{PROXY_NAME}/toxics/{TOXIC_NAME}",
            timeout=5,
        )
    except Exception:
        pass


@contextmanager
def edge_latency_fixture(latency_ms=700, jitter_ms=100):
    # setup
    r = requests.post(
        f"{TOXI_URL}/proxies/{PROXY_NAME}/toxics",
        json={
            "name": TOXIC_NAME,
            "type": "latency",
            "stream": "downstream",  # litellm -> client
            "toxicity": 1.0,
            "attributes": {"latency": latency_ms, "jitter": jitter_ms},
        },
        timeout=5,
    )
    r.raise_for_status()
    try:
        yield
    finally:
        # teardown
        remove_toxic_if_present()


# Cleanup
remove_toxic_if_present()

# Warmup
for i in range(3):
    time_chat = timed_chat(BASE_URL_CLEAN)
    print(
        f"[pre-flight #{i} to start model] duration={time_chat[0]:.2f}s\n{time_chat[1]}\n"
    )

clean_s, clean_text = timed_chat(BASE_URL_CHAOS)  # same ingress, no toxic
print(f"[baseline] duration={clean_s:.2f}s\n{clean_text}\n")

baseline_latency_ms = int(clean_s * 1000) + 500
print(f"Latency based on baseline response time: {baseline_latency_ms} ms")
with edge_latency_fixture(latency_ms=baseline_latency_ms, jitter_ms=100):
    chaos_s, chaos_text = timed_chat(BASE_URL_CHAOS)
    print(f"[chaos] duration={chaos_s:.2f}s\n{chaos_text}\n")

# notebook assertions
assert chaos_text is not None
assert chaos_s > clean_s + 0.4, (
    f"Expected clear delay. clean={clean_s:.2f}s chaos={chaos_s:.2f}s"
)

cleanup_s, cleanup_text = timed_chat(BASE_URL_CHAOS)  # Ensure we have no delay
print(f"[post-flight to verify clear line] duration={cleanup_s:.2f}s\n{cleanup_text}\n")
assert cleanup_text is not None
assert cleanup_s < chaos_s

print("OK: edge latency toxic had measurable effect and was cleaned up.")

[pre-flight #0 to start model] duration=0.76s
Hallo! Wie kann ich Ihnen helfen?

[pre-flight #1 to start model] duration=0.93s
Hallo! Wie kann ich dir helfen?

[pre-flight #2 to start model] duration=0.94s
Hallo! Wie kann ich Ihnen helfen?

[baseline] duration=0.91s
Hallo! Wie kann ich Ihnen helfen?

Latency based on baseline response time: 1413 ms
[chaos] duration=1.82s
Hallo!

[post-flight to verify clear line] duration=1.19s
Hallo! Wie kann ich Ihnen helfen?

OK: edge latency toxic had measurable effect and was cleaned up.
